In [1]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.stats import chi2_contingency
from itertools import combinations


In [2]:
df = pd.read_csv(r'C:\Users\hp\Desktop\QBC-miniv1\social_anxiety_dataset.csv')

In [3]:
df.describe()


,Age,Sleep Hours,Physical Activity (hrs/week),Caffeine Intake (mg/day),Alcohol Consumption (drinks/week),Stress Level (1-10),Heart Rate (bpm),Breathing Rate (breaths/min),Sweating Level (1-5),Therapy Sessions (per month),Diet Quality (1-10),Anxiety Level (1-10),Target,is_Anxious
count,1968.000000,1994.000000,2030.000000,1936.000000,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,2030.000000,1952.000000,2030.000000,2030.000000,2030.000000
mean,39.921748,6.424624,2.800246,306.275826,9.490148,6.028571,93.066502,20.955665,3.095567,2.359113,5.234119,3.922660,0.103448,0.103448
std,13.243754,2.161544,2.233899,208.012717,6.028653,3.137269,23.313430,5.182524,1.392697,2.148642,2.884946,2.139176,0.304619,0.304619
min,18.000000,-10.000000,-10.000000,0.000000,-10.000000,1.000000,60.000000,12.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,29.000000,5.800000,1.400000,177.750000,5.000000,3.000000,76.000000,17.000000,2.000000,1.000000,3.000000,2.000000,0.000000,0.000000
50%,40.000000,6.700000,2.800000,277.500000,10.000000,6.000000,93.000000,21.000000,3.000000,2.000000,5.000000,4.000000,0.000000,0.000000
75%,51.000000,7.500000,4.200000,391.000000,15.000000,9.000000,107.000000,26.000000,4.000000,3.000000,8.000000,5.000000,0.000000,0.000000
max,64.000000,11.000000,9.200000,1500.000000,19.000000,15.000000,220.000000,29.000000,5.000000,10.000000,10.000000,10.000000,1.000000,1.000000


In [4]:
df.head()

,Age,Gender,Occupation,Sleep Hours,Physical Activity (hrs/week),Caffeine Intake (mg/day),Alcohol Consumption (drinks/week),Smoking,Family History of Anxiety,Stress Level (1-10),...,Sweating Level (1-5),Dizziness,Medication,Therapy Sessions (per month),Recent Major Life Event,Diet Quality (1-10),Anxiety Level (1-10),Target,is_Anxious,Therapy History
0,59.0,Other,Teacher,7.0,2.4,40.0,5,Yes,No,4,...,5,No,No,0,Yes,1.0,2.0,0,0,Group Therapy
1,46.0,Female,Student,5.1,5.4,156.0,11,NaN,No,3,...,4,Yes,No,2,No,1.0,4.0,0,0,NaN
2,40.0,Other,Lawyer,5.1,1.9,570.0,14,Yes,Yes,9,...,3,No,No,6,Yes,4.0,9.0,1,1,NaN
3,40.0,Male,Nurse,7.6,0.9,129.0,0,No,No,9,...,2,No,NaN,2,No,2.0,6.0,0,0,NaN
4,26.0,Male,Other,6.7,3.0,64.0,13,No,No,15,...,4,No,Yes,0,Yes,4.0,3.0,0,0,NaN


In [ ]:
def shapiro_test(data, cl):
    a = data[cl]
    shapiro_stat, shapiro_p = stats.shapiro(a)
    
    print(f"Normality Test")
    print("-" * 20)
    print(f"Shapiro-Wilk: Statistic = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}") 
    
    #h0 
    alpha = 0.05
    if shapiro_p > alpha:
        print("accept h0")
    else:
        print("reject h0")


In [ ]:
def one_sample_ttest(data, cl, mu0):
    a = data[cl]
    n = len(a)
    mean = np.mean(a)
    s = np.std(a, ddof=1)
    se = s / np.sqrt(n)
    t_stat = (mean - mu0) / se
    df = n - 1
    p_valeu = stats.t.cdf(t_stat, df)

    t_critical = stats.t.ppf(0.975, df)
    margin = t_critical * se
    ci_lower = mean - margin
    ci_upper = mean + margin


    print(f"One-Sample t-Test")
    print("-" * 20)
    print(f"n = {n}")
    print(f"x̄ = {mean:.3f}")
    print(f"se = {se:.4f}")
    print(f"t = {t_stat:.4f}")
    print(f"p-value = {p_valeu:.4f}")
    print(f"CI 95% = [{ci_lower:.3f}, {ci_upper:.3f}]")
    
    if p_valeu < 0.05:
        print("reject h0")
    else:
        print("accept h0")


In [ ]:

def two_sample_ttest(df, cl_num, cl_target, data1, data2):

    group1 = df[df[cl_target] == data1][cl_num]
    group2 = df[df[cl_target] == data2][cl_num]
    
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = np.mean(group1), np.mean(group2)
    var_A = ((group1 - mean1)** 2).sum() / (len(group1)-1)
    var_B = ((group2 - mean2)** 2).sum() / (len(group2)-1)
    sp = np.sqrt(((n1 - 1 )* var_A + (n2 - 1) * var_B)/ (n1 + n2 - 2))
    
    t_stat = (mean1 - mean2) / (sp * np.sqrt(1/n1 + 1/n2))

    df = n1 + n2 - 2
    
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat),df))

    t_critical = stats.t.ppf(0.975, df)
    margin = t_critical * sp * np.sqrt(1/n1 + 1/n2)
    ci_lower = (mean1 - mean2) - margin
    ci_upper = (mean1 - mean2) + margin


    print(f"two_sample_ttest")
    print("-" * 20)
    print(f"1mean": {mean1:.3f}"")
    print(f"2mean: {mean2:.3f}")
    print(f"t = {t_stat:.4f}")
    print(f"p-value = {p_value:.4f}")
    
    if p_value < 0.05:
        print("reject h0")
    else:
        print("accept h0")

In [ ]:
def chi_2_test(data, cl1, cl2):

    contingency_table = pd.crosstab(data[cl1], data[cl2])
    
    chi2_stat, p_value, dof, expected_table = stats.chi2_contingency(contingency_table)
    
    print(f"Chi-2 Test")
    print("-" * 20)
    print(f"Chi-2= {chi2_stat:.4f}")
    print(f"p-value = {p_value:.4f}")
    print(f"df = {dof}")
    
    if p_value < 0.05:
        print("reject h0")
    else:
        print("accept (mostaghel)")


In [ ]:
def mann_whitney(data1, data2, alternative='two-sided'):
    u_stat, p_value = stats.mannwhitneyu(data1, data2, alternative=alternative)
    
    print(f"U statistic = {u_stat:.4f}")
    print(f"p-value = {p_value:.4f}")

In [ ]:
def tasmim(df, cl_num, cl_target, data1, data2):
    group1 = df[df[cl_target] == data1][cl_num]
    group2 = df[df[cl_target] == data2][cl_num]

    t1, p1 = stats.shapiro(group1)
    t2, p2 = stats.shapiro(group2)

    print(f"normality test: {p1:.4f},  {p2:.4f}")

    if p1> 0.05 and p2>0.05 :
        print("use ttest")
        print("-"*20)
        two_sample_ttest(df, cl_num, cl_target, data1, data2)

    else:
        print("use mann_whitney")

In [ ]:

def correlation(data, cl1, cl2):
    group1 = data[cl1]
    group2 = data[cl2]

    t1, p1 = stats.shapiro(group1)
    t2, p2 = stats.shapiro(group2)

    print(f"normality test: {p1:.4f},  {p2:.4f}")

    if p1> 0.05 and p2>0.05 :
        corr_value, p_value = stats.pearsonr(group1, group2)
        name = 'pearson'

    else:
        corr_value, p_value = stats.spearmanr(group1, group2)
        name = 'spearman'

    print(f"{name}")
    print(f"correlation : {corr_value:.4f}")
    print(f"p_value: {p_value:.4f}")

بررسی نرمال بودن داده های عددی

آزمون های تک نمونه ای  :
Anxiety Level 
sleep hour
stress level
coffeine intake

آزمون های دو نمونه ای:
جنسیت و اضطراب
سیگار کشیدن و استرس
خواب و لول اضطراب
رژیم و لول اضطراب
شاخص های جدید
....

آزمون های کای دو:

آزمون های همبستگی: